# Modelo final: codificación, modelos, umbral, evaluación en prueba e interpretabilidad

Continúa la cadena de notebooks: carga el checkpoint de `issue6.ipynb` (el Issue 7 concluyó que PCA no se aplica, así que el dataset final es el mismo). Reúne en un solo notebook los pasos de la etapa de modelado:

1. Codificación de las 8 categóricas y decisión sobre `loan_amnt`.
2. Comparación en validación: regresión logística, Random Forest y gradient boosting (`HistGradientBoostingClassifier`).
3. Umbral y calibración, elegidos en validación.
4. **Evaluación en prueba, una sola vez**, del modelo final y de la regresión logística.
5. Comparación con la calificación `grade` de Lending Club, interpretabilidad y análisis de errores.

**Reglas:** todo se ajusta con entrenamiento y se elige en validación. El conjunto de prueba solo se usa en la sección 6. `issue_d` no es predictor: solo se usa como fecha de referencia para calcular la antigüedad del historial crediticio y para el análisis por año.

**Librerías:** solo `scikit-learn`, que ya está en `requirements.txt`. `HistGradientBoostingClassifier` implementa gradient boosting por histogramas, el mismo algoritmo de LightGBM.

In [ ]:
# Carga el checkpoint de issue6 (data/checkpoints/issue6/) en lugar de ejecutar toda la cadena.
import sys
sys.path.append('..')
from src.checkpoints import cargar_checkpoint, existe_checkpoint

if existe_checkpoint('issue6'):
    globals().update(cargar_checkpoint('issue6'))
else:
    %run issue7.ipynb

In [ ]:
import time
import warnings
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.calibration import calibration_curve
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.exceptions import ConvergenceWarning
from sklearn.inspection import PartialDependenceDisplay, permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (average_precision_score, brier_score_loss, confusion_matrix,
                             precision_recall_curve, precision_score, recall_score, f1_score,
                             roc_auc_score, roc_curve)
from sklearn.preprocessing import OneHotEncoder

from src.data import COLUMNA_FECHA, SEED, cargar_prestamos, ruta_datos, ESTADOS_TARGET

pd.set_option('display.width', 200)
pd.set_option('display.max_rows', 120)

DIR_FIGURAS = Path.cwd().parent / 'reports' / 'figures'
DIR_MODELOS = Path.cwd().parent / 'data' / 'modelos'
AZUL, NARANJA, VERDE, TINTA, TINTA_2, REJILLA = '#2a78d6', '#eb6834', '#2f9e6e', '#0b0b0b', '#52514e', '#e4e3df'
plt.rcParams.update({
    'figure.facecolor': 'white', 'axes.facecolor': 'white', 'axes.edgecolor': TINTA_2,
    'axes.labelcolor': TINTA, 'xtick.color': TINTA_2, 'ytick.color': TINTA_2,
    'axes.grid': True, 'grid.color': REJILLA, 'grid.linewidth': 0.8,
    'axes.spines.top': False, 'axes.spines.right': False, 'font.size': 10,
    'axes.titlesize': 11, 'axes.titleweight': 'bold',
})

def guardar(fig, nombre):
    fig.savefig(DIR_FIGURAS / nombre, dpi=150, bbox_inches='tight')
    print(f'Figura guardada en reports/figures/{nombre}')

## 1. Datos

El checkpoint trae 109 numéricas (61 continuas tratadas por el Issue 6 y 48 indicadores de nulo del Issue 5) y 8 categóricas sin codificar. `issue_d` no está en `X`: se toma de la base común (`src/data.py`) por índice, solo como fecha de referencia.

In [ ]:
base = cargar_prestamos()
fecha = {nombre: base.loc[X.index, COLUMNA_FECHA] for nombre, X in
         (('train', X_train), ('val', X_val), ('test', X_test))}
for nombre, X, y in (('train', X_train, y_train), ('val', X_val, y_val), ('test', X_test, y_test)):
    assert (base.loc[X.index, 'target'].values == y.values).all(), f'{nombre}: índices desalineados'

categoricas = [c for c in X_train.columns if X_train[c].dtype == object]
numericas = [c for c in X_train.columns if c not in categoricas]
print(f'Numéricas: {len(numericas)}; categóricas: {len(categoricas)} {categoricas}')
print(f'Entrenamiento {X_train.shape}, validación {X_val.shape}, prueba {X_test.shape}')
print(f'Charged Off: {y_train.mean():.4f} / {y_val.mean():.4f} / {y_test.mean():.4f}')

## 2. Codificación de las categóricas

| Columna | Tratamiento | Motivo |
|---|---|---|
| `term` | Binaria: 1 si es a 60 meses | Solo dos valores. |
| `emp_length` | Años de 0 a 10 (`< 1 year` = 0, `10+ years` = 10) | Es ordinal: one-hot perdería el orden. El Issue 5 ya imputó sus nulos y dejó su indicador. |
| `earliest_cr_line` | Antigüedad del historial crediticio en meses, respecto de `issue_d` | Como fecha absoluta no generaliza a solicitudes futuras; la antigüedad sí. `issue_d` es la mejor aproximación disponible a la fecha de la solicitud y aquí solo se usa como referencia, no como predictor. |
| `home_ownership`, `verification_status`, `purpose`, `addr_state`, `application_type` | One-hot | Las categorías se aprenden en entrenamiento. Las que tienen menos de 500 préstamos (por ejemplo, `home_ownership` = `ANY`, `NONE`, `OTHER`) se agrupan en una categoría "infrecuente", y las desconocidas en validación o prueba caen ahí. |

Las tres numéricas nuevas se estandarizan con la media y la desviación de entrenamiento, igual que el resto de la cadena.

In [ ]:
ONE_HOT = ['home_ownership', 'verification_status', 'purpose', 'addr_state', 'application_type']

def anios_empleo(s):
    s = s.str.strip()
    return (s.replace({'< 1 year': '0', '10+ years': '10'})
             .str.extract(r'(\d+)', expand=False).astype(float))

def meses_historial(earliest, ref):
    inicio = pd.to_datetime(earliest, format='%b-%Y')
    meses = (ref.dt.year - inicio.dt.year) * 12 + (ref.dt.month - inicio.dt.month)
    return meses.clip(lower=0).astype(float)

def numericas_nuevas(X, ref):
    return pd.DataFrame({
        'term_60': (X['term'].str.strip() == '60 months').astype(float),
        'emp_length_anios': anios_empleo(X['emp_length']),
        'historial_meses': meses_historial(X['earliest_cr_line'], ref),
    }, index=X.index)

codificador = OneHotEncoder(handle_unknown='infrequent_if_exist', min_frequency=500,
                            sparse_output=False, dtype=np.float32).fit(X_train[ONE_HOT])
nuevas_train = numericas_nuevas(X_train, fecha['train'])
media_nuevas, desv_nuevas = nuevas_train.mean(), nuevas_train.std()

def codificar(X, ref):
    nuevas = (numericas_nuevas(X, ref) - media_nuevas) / desv_nuevas
    oh = pd.DataFrame(codificador.transform(X[ONE_HOT]), index=X.index,
                      columns=codificador.get_feature_names_out(ONE_HOT))
    return pd.concat([X[numericas].astype(np.float32), nuevas.astype(np.float32), oh], axis=1)

Xc_train, Xc_val, Xc_test = (codificar(X, fecha[n]) for X, n in
                             ((X_train, 'train'), (X_val, 'val'), (X_test, 'test')))
assert list(Xc_train.columns) == list(Xc_val.columns) == list(Xc_test.columns)
assert not any(X.isna().any().any() for X in (Xc_train, Xc_val, Xc_test))

columnas_oh = list(codificador.get_feature_names_out(ONE_HOT))
print(f'Columnas finales: {Xc_train.shape[1]} = {len(numericas)} numéricas + 3 nuevas + {len(columnas_oh)} one-hot')
print('Categorías agrupadas como infrecuentes:',
      {c: list(v) for c, v in zip(ONE_HOT, codificador.infrequent_categories_) if v is not None})
nuevas_train.describe().round(1)

## 3. Regresión logística: categóricas y `loan_amnt`

Misma configuración que en los issues anteriores (`max_iter=1000`, `random_state=SEED`). Se comparan en validación:

- Solo numéricas: la referencia de los issues 6 y 7 (ROC-AUC 0.6883, PR-AUC 0.3468).
- Numéricas + categóricas codificadas.
- Lo mismo sin `loan_amnt`, para decidir si esa variable condicional se queda.

In [ ]:
def metricas(y, prob):
    return {'ROC-AUC': roc_auc_score(y, prob), 'PR-AUC': average_precision_score(y, prob)}

def ajustar_logistica(X_a, X_b, **kw):
    modelo = LogisticRegression(max_iter=1000, random_state=SEED, **kw)
    inicio = time.time()
    with warnings.catch_warnings(record=True) as avisos:
        warnings.simplefilter('always', ConvergenceWarning)
        modelo.fit(X_a, y_train)
    prob = modelo.predict_proba(X_b)[:, 1]
    return modelo, prob, {**metricas(y_val, prob), 'segundos': round(time.time() - inicio, 1),
                          'convergió': not avisos}

sin_loan = [c for c in Xc_train.columns if c != 'loan_amnt']
resultados = []
for nombre, cols in (('Logística: solo numéricas', numericas),
                     ('Logística: numéricas + categóricas', list(Xc_train.columns)),
                     ('Logística: + categóricas, sin loan_amnt', sin_loan)):
    modelo, prob, res = ajustar_logistica(Xc_train[cols], Xc_val[cols])
    resultados.append({'modelo': nombre, 'columnas': len(cols), **res})
    if nombre == 'Logística: numéricas + categóricas':
        logistica, prob_val_logistica = modelo, prob
pd.DataFrame(resultados).set_index('modelo').round(4)

**Lectura:**

- **Las categóricas aportan bastante.** La logística pasa de 0.6883 a 0.7179 de ROC-AUC y de 0.3468 a 0.3891 de PR-AUC. Es la mayor mejora de toda la cadena desde el baseline: `term`, `purpose`, `home_ownership` y `addr_state` traen información que ninguna numérica tenía.
- **`loan_amnt` aporta poco, pero algo.** Sin ella, el ROC-AUC baja 0.0055 (0.7179 → 0.7124). Se conserva en el modelo final porque la cadena la incluye desde el Issue 2, pero sigue siendo una variable condicional de la propuesta (sección 7): el diccionario de Lending Club dice que el monto puede reflejar reducciones del área de crédito. **Es una decisión que el equipo debe confirmar** (ver conclusiones).

## 4. Modelos de árboles

**Gradient boosting (`HistGradientBoostingClassifier`).** Búsqueda aleatoria de 10 configuraciones con semilla fija. Cada una usa early stopping con un 10 % de entrenamiento apartado internamente (no con validación), así que validación queda limpia para elegir la configuración por ROC-AUC.

**Random Forest.** Una configuración razonable (300 árboles, hojas de al menos 50 préstamos, `max_features='sqrt'`). Cada árbol se entrena con un 25 % de las filas (`max_samples`) para que el tiempo sea manejable con 862,757 préstamos.

Los árboles no necesitan escalado ni `log1p`; usan las mismas columnas que la logística porque esas transformaciones son monótonas y no cambian sus cortes.

In [ ]:
rng = np.random.default_rng(SEED)
ESPACIO = {
    'learning_rate': [0.03, 0.05, 0.1],
    'max_leaf_nodes': [31, 63, 127],
    'min_samples_leaf': [100, 500, 2000],
    'l2_regularization': [0.0, 1.0, 10.0],
    'max_features': [0.5, 0.8, 1.0],
}
configuraciones = [{k: v[rng.integers(len(v))] for k, v in ESPACIO.items()} for _ in range(10)]

busqueda = []
for i, conf in enumerate(configuraciones, 1):
    modelo = HistGradientBoostingClassifier(max_iter=1000, early_stopping=True, validation_fraction=0.1,
                                            n_iter_no_change=50, random_state=SEED, **conf)
    inicio = time.time()
    modelo.fit(Xc_train, y_train)
    prob = modelo.predict_proba(Xc_val)[:, 1]
    busqueda.append({**conf, 'iteraciones': modelo.n_iter_, **metricas(y_val, prob),
                     'segundos': round(time.time() - inicio, 1), '_modelo': modelo, '_prob': prob})
    print(f"{i:>2}/10  ROC-AUC {busqueda[-1]['ROC-AUC']:.4f}  PR-AUC {busqueda[-1]['PR-AUC']:.4f}  "
          f"{modelo.n_iter_} iteraciones, {busqueda[-1]['segundos']} s")

tabla_busqueda = pd.DataFrame(busqueda).drop(columns=['_modelo', '_prob']).sort_values('ROC-AUC', ascending=False)
mejor_gb = max(busqueda, key=lambda r: r['ROC-AUC'])
gb, prob_val_gb = mejor_gb['_modelo'], mejor_gb['_prob']
tabla_busqueda.round(4)

In [ ]:
inicio = time.time()
rf = RandomForestClassifier(n_estimators=300, min_samples_leaf=50, max_features='sqrt', max_samples=0.25,
                            n_jobs=-1, random_state=SEED).fit(Xc_train, y_train)
prob_val_rf = rf.predict_proba(Xc_val)[:, 1]
segundos_rf = round(time.time() - inicio, 1)
print(f'Random Forest: {metricas(y_val, prob_val_rf)}, {segundos_rf} s')

In [ ]:
comparacion = pd.DataFrame(resultados + [
    {'modelo': 'Random Forest', 'columnas': Xc_train.shape[1], **metricas(y_val, prob_val_rf), 'segundos': segundos_rf},
    {'modelo': 'Gradient boosting (mejor de 10)', 'columnas': Xc_train.shape[1], **metricas(y_val, prob_val_gb),
     'segundos': mejor_gb['segundos']},
]).set_index('modelo')[['columnas', 'ROC-AUC', 'PR-AUC', 'segundos']]

fig, ejes = plt.subplots(1, 2, figsize=(13, 3.6))
for ax, metrica in zip(ejes, ['ROC-AUC', 'PR-AUC']):
    valores = comparacion[metrica]
    colores = [NARANJA if 'boosting' in m else (VERDE if 'Forest' in m else AZUL) for m in valores.index]
    ax.barh(valores.index[::-1], valores.values[::-1], color=colores[::-1])
    for j, v in enumerate(valores.values[::-1]):
        ax.text(v, j, f' {v:.4f}', va='center', fontsize=8)
    ax.set_xlim(valores.min() - 0.02, valores.max() + 0.015)
    ax.set_title(f'{metrica} en validación')
    ax.grid(axis='y', visible=False)
fig.tight_layout()
guardar(fig, 'modelo_comparacion_validacion.png')
plt.show()
comparacion.round(4)

**Lectura:**

- **El gradient boosting es el mejor modelo:** ROC-AUC 0.7338 y PR-AUC 0.4105 en validación, frente a 0.7179 y 0.3891 de la logística con las mismas columnas. La ganancia (+0.016 de ROC-AUC, +0.021 de PR-AUC) viene de las interacciones y no linealidades que la logística no captura.
- **La búsqueda es estable:** las 10 configuraciones quedan entre 0.7314 y 0.7338. Ganan las de tasa de aprendizaje baja (0.03–0.05), hojas grandes (`min_samples_leaf` = 2000) y árboles de 63–127 hojas; las de `min_samples_leaf` = 100 quedan últimas, lo que sugiere que hojas pequeñas sobreajustan. Más búsqueda difícilmente movería el resultado más de unas milésimas.
- **El Random Forest no supera a la logística** (0.7156). Con hojas de 50 préstamos y `max_features='sqrt'` cada árbol ve pocas variables por corte; el boosting, que corrige errores de forma secuencial, aprovecha mejor una señal débil y repartida como esta.
- **Modelo elegido:** gradient boosting con `learning_rate` 0.03, `max_leaf_nodes` 127, `min_samples_leaf` 2000, `l2_regularization` 0, `max_features` 0.8 (876 iteraciones).

## 5. Umbral y calibración (validación)

Se elige el umbral en validación con el mismo criterio del baseline: el que maximiza F1 de `Charged Off`. Se muestran también otros puntos de la curva, porque el umbral es una decisión de negocio (cuánto cuesta un incumplimiento no detectado frente a rechazar a un buen pagador), no una propiedad del modelo.

In [ ]:
MODELO_FINAL = 'Gradient boosting'
modelo_final, prob_val_final = gb, prob_val_gb

def umbral_f1(y, prob):
    p, r, t = precision_recall_curve(y, prob)
    f1 = 2 * p * r / np.clip(p + r, 1e-12, None)
    i = np.nanargmax(f1[:-1])
    return float(t[i])

umbral = umbral_f1(y_val, prob_val_final)
umbral_logistica = umbral_f1(y_val, prob_val_logistica)

def al_umbral(y, prob, t):
    pred = (prob >= t).astype(int)
    return {'umbral': t, 'recall': recall_score(y, pred), 'precisión': precision_score(y, pred),
            'F1': f1_score(y, pred), '% marcados': pred.mean() * 100}

puntos = [al_umbral(y_val, prob_val_final, t) for t in (0.10, 0.15, umbral, 0.25, 0.30, 0.40)]
print(f'Umbral elegido (máximo F1 en validación): {umbral:.4f}; logística: {umbral_logistica:.4f}')
pd.DataFrame(puntos).round(4)

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
for nombre, prob, color in ((MODELO_FINAL, prob_val_final, NARANJA), ('Logística', prob_val_logistica, AZUL)):
    frac, media = calibration_curve(y_val, prob, n_bins=15, strategy='quantile')
    ax.plot(media, frac, marker='o', markersize=3, color=color,
            label=f'{nombre} (Brier {brier_score_loss(y_val, prob):.4f})')
ax.plot([0, 0.8], [0, 0.8], color=TINTA_2, linestyle='--', linewidth=1)
ax.set_xlabel('Probabilidad predicha (media del bin)')
ax.set_ylabel('Tasa observada de Charged Off')
ax.set_title('Calibración en validación')
ax.legend(frameon=False, fontsize=8)
fig.tight_layout()
guardar(fig, 'modelo_calibracion.png')
plt.show()

**Lectura:**

- **Umbral elegido: 0.2234.** En validación detecta el 61.4 % de los `Charged Off` con una precisión de 35.4 %, y marca como riesgosos al 34.7 % de los préstamos. Bajar el umbral a 0.15 sube el recall a 80 % pero marca a más de la mitad de los préstamos; subirlo a 0.40 da precisión de 49 % con solo 25 % de recall. La tabla sirve para elegir otro punto si se fija un costo explícito.
- **Ambos modelos están bien calibrados:** las curvas siguen la diagonal y el Brier es 0.1411 (boosting) y 0.1437 (logística). No hace falta recalibrar: una probabilidad predicha de 0.30 corresponde a una tasa observada de `Charged Off` cercana al 30 %.

## 6. Evaluación en prueba (una sola vez)

Con el modelo, sus hiperparámetros y el umbral ya fijados en validación, se evalúan por única vez en prueba el modelo final y la regresión logística con las mismas columnas. El baseline de la propuesta (regresión logística con 9 variables, `02_baseline_sin_leakage.ipynb`) obtuvo en prueba ROC-AUC 0.6331 y PR-AUC 0.2848.

In [ ]:
prob_test_final = modelo_final.predict_proba(Xc_test)[:, 1]
prob_test_logistica = logistica.predict_proba(Xc_test)[:, 1]

filas = []
for nombre, prob_v, prob_t, t in ((MODELO_FINAL, prob_val_final, prob_test_final, umbral),
                                  ('Logística (numéricas + categóricas)', prob_val_logistica, prob_test_logistica, umbral_logistica)):
    for particion, y, prob in (('validación', y_val, prob_v), ('prueba', y_test, prob_t)):
        filas.append({'modelo': nombre, 'partición': particion, **metricas(y, prob), **al_umbral(y, prob, t)})
filas.append({'modelo': 'Baseline de la propuesta (9 variables)', 'partición': 'prueba',
              'ROC-AUC': 0.6331, 'PR-AUC': 0.2848, 'umbral': 0.1831, 'recall': 0.7324, 'precisión': 0.2498})
evaluacion = pd.DataFrame(filas).set_index(['modelo', 'partición'])
evaluacion.round(4)

In [ ]:
tn, fp, fn, tp = confusion_matrix(y_test, (prob_test_final >= umbral).astype(int)).ravel()
print(f'{MODELO_FINAL} en prueba, umbral {umbral:.4f}:')
print(f'  Charged Off detectados (VP): {tp:,} de {tp + fn:,}  |  no detectados (FN): {fn:,}')
print(f'  Pagados marcados como riesgo (FP): {fp:,} de {tn + fp:,}  |  aprobados correctamente (VN): {tn:,}')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
for nombre, prob, color in ((MODELO_FINAL, prob_test_final, NARANJA), ('Logística', prob_test_logistica, AZUL)):
    fpr, tpr, _ = roc_curve(y_test, prob)
    ax1.plot(fpr, tpr, color=color, label=f'{nombre} ({roc_auc_score(y_test, prob):.4f})')
    p, r, _ = precision_recall_curve(y_test, prob)
    ax2.plot(r, p, color=color, label=f'{nombre} ({average_precision_score(y_test, prob):.4f})')
ax1.plot([0, 1], [0, 1], color=TINTA_2, linestyle='--', linewidth=1)
ax2.axhline(y_test.mean(), color=TINTA_2, linestyle='--', linewidth=1)
ax2.text(1, y_test.mean(), f'sin discriminación ({y_test.mean():.2f}) ', ha='right', va='bottom', fontsize=8, color=TINTA_2)
ax1.set(xlabel='Tasa de falsos positivos', ylabel='Tasa de verdaderos positivos (recall)', title='Curva ROC en prueba')
ax2.set(xlabel='Recall', ylabel='Precisión', title='Curva precisión-recall en prueba')
for ax in (ax1, ax2):
    ax.legend(frameon=False, fontsize=8, loc='lower right' if ax is ax1 else 'upper right')
fig.tight_layout()
guardar(fig, 'modelo_curvas_prueba.png')
plt.show()

**Lectura:**

- **Prueba confirma validación.** El gradient boosting obtiene ROC-AUC 0.7354 y PR-AUC 0.4122 en prueba (0.7338 y 0.4105 en validación). La diferencia es mínima y positiva: la selección de hiperparámetros y del umbral no sobreajustó validación.
- **Frente al baseline de la propuesta** (logística con 9 numéricas: 0.6331 / 0.2848), el modelo final gana 0.102 de ROC-AUC y 0.127 de PR-AUC. La PR-AUC de 0.412 duplica la referencia sin discriminación (0.20).
- **Al umbral 0.2234**, en prueba detecta 33,235 de los 53,864 incumplimientos (recall 61.7 %) y marca a 60,510 buenos pagadores (28 % de ellos). De cada 100 préstamos marcados, 35 terminan en `Charged Off`, frente a 20 si se marcaran al azar.
- La logística con categóricas queda cerca (0.7185 / 0.3906): es una alternativa razonable si se prioriza la interpretabilidad.

## 7. Comparación con la calificación de Lending Club

`grade` (A a G) no es predictor: la asigna Lending Club después de evaluar la solicitud, con información y criterios que el modelo no tiene, y la propuesta la excluye por eso. Sí sirve como **referencia**: su ROC-AUC sobre los mismos préstamos de prueba mide qué tan bien ordena el riesgo la evaluación de Lending Club. Se lee del archivo original solo para esta comparación.

In [ ]:
estados = pd.read_csv(ruta_datos(), usecols=['loan_status', 'grade'], low_memory=False)
estados = estados.loc[estados['loan_status'].isin(ESTADOS_TARGET.keys())].reset_index(drop=True)
assert len(estados) == len(base) and (estados['loan_status'].map(ESTADOS_TARGET).values == base['target'].values).all()
grade_test = estados.loc[X_test.index, 'grade'].map({g: i for i, g in enumerate('ABCDEFG', 1)})

referencia_lc = pd.DataFrame([
    {'ordenamiento': 'grade de Lending Club (referencia, no predictor)', **metricas(y_test, grade_test)},
    {'ordenamiento': MODELO_FINAL, **metricas(y_test, prob_test_final)},
    {'ordenamiento': 'Logística (numéricas + categóricas)', **metricas(y_test, prob_test_logistica)},
]).set_index('ordenamiento')
print('Tasa de Charged Off por grade en prueba:')
print(y_test.groupby(grade_test.map(dict(enumerate('ABCDEFG', 1)))).mean().round(3).to_dict())
referencia_lc.round(4)

**Lectura:**

- La calificación de Lending Club ordena bien el riesgo (la tasa de `Charged Off` sube de 5.9 % en A a 48.9 % en G), pero su ROC-AUC sobre estos préstamos es 0.6797, **por debajo del modelo (0.7354) e incluso de la logística (0.7185)**.
- Esto **no** significa que el modelo sea mejor que el proceso de Lending Club. `grade` tiene solo 7 niveles, así que ordena con muchos empates (`sub_grade`, con 35 niveles, ordenaría mejor), y el dataset solo contiene préstamos que Lending Club ya aprobó: las solicitudes que su evaluación rechazó no están. Lo que sí muestra es que, con información disponible antes de la aprobación, el modelo recupera al menos tanta capacidad de ordenar el riesgo como la calificación asignada después.

## 8. Interpretabilidad

`HistGradientBoostingClassifier` no tiene importancias propias. Se usa **importancia por permutación**: cuánto cae el ROC-AUC en validación al desordenar una variable. Se calcula sobre una muestra fija de 50,000 préstamos de validación, con 3 repeticiones. Las columnas one-hot de una misma variable se reportan agregadas sumando sus caídas.

Después, la **dependencia parcial** de las variables más importantes muestra cómo cambia la probabilidad predicha al moverlas. Los ejes están en unidades estandarizadas (y en logaritmo cuando el Issue 6 aplicó `log1p`), porque así entran al modelo.

In [ ]:
muestra = Xc_val.sample(50_000, random_state=SEED)
inicio = time.time()
perm = permutation_importance(modelo_final, muestra, y_val.loc[muestra.index], scoring='roc_auc',
                              n_repeats=3, random_state=SEED, n_jobs=-1)
print(f'Permutación: {time.time() - inicio:.0f} s')

importancia = pd.Series(perm.importances_mean, index=muestra.columns)
grupo = {c: next((v for v in ONE_HOT if c.startswith(v + '_')), c) for c in importancia.index}
importancia_agr = importancia.groupby(grupo).sum().sort_values(ascending=False)

top = importancia_agr.head(15)
fig, ax = plt.subplots(figsize=(7, 4.6))
ax.barh(top.index[::-1], top.values[::-1], color=AZUL)
ax.set_xlabel('Caída de ROC-AUC al permutar (validación)')
ax.set_title(f'{MODELO_FINAL}: 15 variables más importantes')
ax.grid(axis='y', visible=False)
fig.tight_layout()
guardar(fig, 'modelo_importancia_permutacion.png')
plt.show()
importancia_agr.head(20).round(4)

In [ ]:
top_numericas = [c for c in importancia_agr.index if c in Xc_train.columns and c not in columnas_oh][:4]
fig, ejes = plt.subplots(1, len(top_numericas), figsize=(4 * len(top_numericas), 3.4), sharey=True)
PartialDependenceDisplay.from_estimator(modelo_final, muestra.sample(10_000, random_state=SEED).astype('float64'), top_numericas,
                                        ax=ejes, response_method='predict_proba', method='brute', grid_resolution=30,
                                        line_kw={'color': NARANJA})
fig.suptitle('Dependencia parcial: probabilidad de Charged Off (ejes en unidades transformadas)', fontweight='bold')
fig.tight_layout()
guardar(fig, 'modelo_dependencia_parcial.png')
plt.show()

**Lectura:**

- **El plazo domina las decisiones.** Los préstamos a 60 meses tienen el doble de incumplimiento (32.4 % frente a 16.0 %), y el modelo marca a casi todos: recall 86 %, pero 64 % de falsos positivos. En los de 36 meses es al revés: recall 46 %. Dentro de cada plazo el ROC-AUC baja (0.715 y 0.693 frente a 0.734 global), así que parte de la capacidad del modelo es separar plazos. Conviene que el informe reporte las métricas por plazo.
- **Por año**, el ROC-AUC se mantiene entre 0.70 y 0.75, sin degradarse en los años recientes. Recall y falsos positivos suben a partir de 2014, junto con la tasa de incumplimiento. Los préstamos de 2018 resueltos son una submuestra seleccionada (sección 10 de la propuesta), así que su 0.746 no debe leerse como desempeño futuro.
- **Por propósito**, `small_business` es el segmento más difícil: mayor tasa de incumplimiento (29 %), el ROC-AUC más bajo (0.694) y 56 % de falsos positivos.

## 9. Análisis de errores (validación)

Al umbral elegido, se compara el recall (incumplimientos detectados) y la tasa de falsos positivos (buenos pagadores marcados) por segmento. Se hace en validación para no seguir mirando la prueba.

In [ ]:
pred_val = (prob_val_final >= umbral).astype(int)
segmentos = pd.DataFrame({
    'y': y_val.values, 'pred': pred_val, 'prob': prob_val_final,
    'plazo': X_val['term'].str.strip().values,
    'propósito': X_val['purpose'].values,
    'año': fecha['val'].dt.year.values,
}, index=X_val.index)

def por_segmento(col, minimo=2000):
    g = segmentos.groupby(col)
    t = pd.DataFrame({
        'préstamos': g.size(),
        'tasa Charged Off': g['y'].mean(),
        'recall': g.apply(lambda d: d.loc[d.y == 1, 'pred'].mean()),
        'tasa FP': g.apply(lambda d: d.loc[d.y == 0, 'pred'].mean()),
        'ROC-AUC': g.apply(lambda d: roc_auc_score(d.y, d.prob) if d.y.nunique() == 2 else np.nan),
    })
    return t[t['préstamos'] >= minimo].round(3)

display(por_segmento('plazo'))
display(por_segmento('año', minimo=1000))
por_segmento('propósito').sort_values('préstamos', ascending=False)

**Lectura:** _pendiente de la figura de dependencia parcial._

## 10. Conclusiones

1. **Modelo final: gradient boosting** (`HistGradientBoostingClassifier`) con las 186 columnas de la cadena más las categóricas codificadas. En prueba: **ROC-AUC 0.7354, PR-AUC 0.4122**; al umbral 0.2234 elegido en validación, recall 61.7 % y precisión 35.5 %.
2. **Mejora frente al baseline de la propuesta:** de 0.6331 a 0.7354 de ROC-AUC y de 0.2848 a 0.4122 de PR-AUC. La mejora se reparte en tres pasos: usar todas las numéricas preparadas en los issues 2–6 (0.633 → 0.688 en validación), añadir las categóricas (→ 0.718) y pasar a gradient boosting (→ 0.734).
3. **Validación y prueba coinciden** (diferencias de 0.002), y las probabilidades están bien calibradas, así que el modelo puede usarse para ordenar el riesgo o para estimar probabilidades.
4. **Variables más importantes:** el plazo (`term`), el monto (`loan_amnt`), el DTI, el FICO y el ingreso declarado, seguidos de las cuentas abiertas en 24 meses y el estado. Todas tienen sentido crediticio y están disponibles al momento de la solicitud según la auditoría de variables; ninguna sugiere fuga de datos.

**Limitaciones y decisiones pendientes:**

- **`term` y `loan_amnt` son las dos variables más importantes y ambas son condicionales en la propuesta** (sección 7): el valor registrado puede reflejar ajustes de Lending Club durante la aprobación. Sin `loan_amnt` la logística pierde 0.0055 de ROC-AUC. El equipo debe decidir si las acepta como datos de la solicitud o si reporta también el modelo sin ellas.
- **Población:** solo préstamos aprobados y con resultado definitivo (sección 10). Las métricas no representan solicitudes rechazadas ni préstamos que siguen `Current`.
- **Comprobación temporal:** no se hizo el corte cronológico de la sección 10. El ROC-AUC por año de emisión (sección 9) es estable entre 0.70 y 0.75, pero eso no reemplaza entrenar con años anteriores y evaluar en posteriores.
- **Búsqueda de hiperparámetros acotada** (10 configuraciones) e **interpretabilidad con permutación y dependencia parcial** en vez de SHAP, para no añadir dependencias fuera de `scikit-learn`.

In [ ]:
DIR_MODELOS.mkdir(parents=True, exist_ok=True)
joblib.dump({'modelo': modelo_final, 'umbral': umbral, 'columnas': list(Xc_train.columns),
             'codificador': codificador, 'media_nuevas': media_nuevas, 'desv_nuevas': desv_nuevas},
            DIR_MODELOS / 'modelo_final.joblib')
print(f'Modelo guardado en data/modelos/modelo_final.joblib (fuera de git)')